# Notebook 03: Evaluation & Error Analysis

This notebook explores the evaluation pipeline in depth:

1. Per-turn metric computation
2. Rule-based norm violation detection
3. Reward model scoring
4. Error analysis: identifying high-risk or low-performing turns
5. Metric correlation analysis

In [ ]:
import sys
sys.path.insert(0, "../src")

from norm_dialogue_framework.data.synthetic_generator import SyntheticScenarioGenerator
from norm_dialogue_framework.agents import BaselineAgent, CandidateSelectorAgent
from norm_dialogue_framework.simulation.dialogue_runner import DialogueRunner
from norm_dialogue_framework.evaluation.evaluator import Evaluator
from norm_dialogue_framework.evaluation.rule_checks import RuleChecker
from norm_dialogue_framework.evaluation.reward_model import RewardModel
from norm_dialogue_framework.agents.base_agent import DialogueState
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

## 1. Rule-based norm checking

The `RuleChecker` applies heuristic pattern matching to detect norm violations.

In [ ]:
checker = RuleChecker()

test_utterances = [
    # Safe, norm-compliant
    "Could you describe what you observed in your own words? Take your time.",
    "I understand this may be difficult. What happened next?",
    "Thank you for sharing that. Is there anything else you remember?",
    
    # Coercive
    "You must tell me exactly what happened. We already know the truth.",
    "Don't lie. Admit what you did.",
    
    # Leading
    "He was aggressive, wasn't it obvious?",
    "Surely you must have noticed something was wrong?",
    
    # Judgmental
    "That was completely unacceptable behaviour on your part.",
]

rows = []
for u in test_utterances:
    r = checker.check(u)
    rows.append({
        "utterance": u[:70],
        "coercion_risk": r.coercion_risk,
        "leading_risk": r.leading_question_risk,
        "empathy": r.empathy_score,
        "neutrality": r.neutrality_score,
        "procedural": r.procedural_fairness_score,
    })

pd.DataFrame(rows)

## 2. Reward model scoring

In [ ]:
gen = SyntheticScenarioGenerator(seed=42)
scenario = gen.generate(n=1)[0]
state = DialogueState(scenario=scenario, trust_level=0.6, stress_level=0.3)

rm = RewardModel(ethical_weight=0.55, utility_weight=0.45)

candidates = [
    "Could you describe what you observed? Take your time.",
    "You must tell me everything right now.",
    "What happened next, in your own words?",
    "I understand. What else can you tell me about the situation?",
    "Don't lie. We already know what occurred.",
]

ranked = rm.rank_candidates(candidates, state)

print("Candidates ranked by composite reward:")
print("=" * 70)
for rank, (utterance, score) in enumerate(ranked, 1):
    print(f"{rank}. [{score:.3f}] {utterance}")

## 3. Run batch evaluation

In [ ]:
gen = SyntheticScenarioGenerator(seed=42)
scenarios = gen.generate(n=10)

agent = BaselineAgent(seed=42)
evaluator = Evaluator()
runner = DialogueRunner(agent=agent, max_turns=6, min_turns=3, seed=42, evaluator=evaluator)

episodes = [runner.run(s) for s in scenarios]
summaries = evaluator.evaluate_batch(episodes)
df = evaluator.summaries_to_dataframe(summaries)
print(f"Evaluated {len(df)} episodes")
df[["episode_id", "scenario_type", "respondent_profile", "composite_score", "ethical_alignment_score", "utility_score"]].head()

## 4. Score distributions

In [ ]:
fig = px.violin(
    df.melt(value_vars=["composite_score", "ethical_alignment_score", "utility_score"], var_name="metric", value_name="score"),
    x="metric", y="score",
    box=True, points="all",
    title="Score Distributions (Baseline Agent, 10 episodes)"
)
fig.update_layout(yaxis_range=[0, 1])
fig.show()

## 5. Error analysis: high-risk episodes

In [ ]:
risk_threshold = 0.2
high_risk = df[
    (df["mean_coercion_risk"] >= risk_threshold) |
    (df["mean_leading_question_risk"] >= risk_threshold)
]
print(f"High-risk episodes (threshold={risk_threshold}): {len(high_risk)}")

risk_cols = ["episode_id", "scenario_type", "respondent_profile", 
             "mean_coercion_risk", "mean_leading_question_risk", "composite_score"]
high_risk[risk_cols].sort_values("mean_coercion_risk", ascending=False)

## 6. Metric correlation analysis

In [ ]:
numeric_cols = [c for c in df.columns if df[c].dtype in ["float64", "int64"]]
corr = df[numeric_cols].corr()

fig2 = px.imshow(
    corr,
    color_continuous_scale="RdBu_r",
    title="Metric Correlation Matrix (Baseline, 10 episodes)",
    text_auto=".2f",
    aspect="auto"
)
fig2.show()

## 7. Profile-level performance

Examining which respondent profiles are hardest to interact with norm-compliantly.

In [ ]:
if "respondent_profile" in df.columns:
    profile_summary = df.groupby("respondent_profile")[
        ["composite_score", "final_trust_level", "mean_coercion_risk"]
    ].mean().round(3)
    print(profile_summary)
    
    fig3 = px.bar(
        profile_summary.reset_index(),
        x="respondent_profile",
        y="composite_score",
        title="Mean Composite Score by Respondent Profile (Baseline)",
        color="respondent_profile"
    )
    fig3.update_layout(yaxis_range=[0, 1], showlegend=False)
    fig3.show()